# EasyRAG Chunking Principles

## Chapter position

This chapter turns prepared documents into retrieval infrastructure. Chunk boundaries, embedding inputs, vector storage, and workspace artifacts all start here.

## Learning question

How do the chunking strategies choose boundaries, and what kind of metadata does each strategy leave behind?

## Success criteria

- compare sliding-window, structured, and semantic chunking on small documents
- inspect the metadata each strategy emits
- see how `select_chunk_strategy()` chooses a default path from document hints

## Source code anchors

- `easyrag.rag.types.ChunkingConfig`
- `easyrag.rag.indexing.chunking_core.sliding_window_chunk`
- `easyrag.rag.indexing.chunking_core.structured_chunk`
- `easyrag.rag.indexing.chunking_core.semantic_chunk`
- `easyrag.rag.indexing.chunking_core.select_chunk_strategy`


## Method principles

- `easyrag.rag.types.ChunkingConfig`: This config object holds the boundary policy for chunking. It controls window sizes, overlaps, section limits, and semantic split thresholds in one place.
- `easyrag.rag.indexing.chunking_core.sliding_window_chunk`: This chunker trusts length first. It slices text into fixed-size windows and uses overlap so adjacent chunks keep enough context to stay retrievable.
- `easyrag.rag.indexing.chunking_core.structured_chunk`: This chunker trusts document structure first. It tries to keep heading-defined sections intact and only falls back to windows when a section grows too large.
- `easyrag.rag.indexing.chunking_core.semantic_chunk`: This chunker trusts sentence-level embedding similarity. It breaks when adjacent sentence meaning shifts or when the running chunk gets too large.
- `easyrag.rag.indexing.chunking_core.select_chunk_strategy`: This selector chooses a default chunking policy from document hints such as file suffix or title. It is a routing decision, not a chunker by itself.


## How the code fits together

The flow in this notebook is document hints -> strategy choice -> chunk boundaries -> metadata comparison. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.indexing import ChunkingConfig
from easyrag.rag.indexing.chunking import (
    select_chunk_strategy,
    semantic_chunk,
    sliding_window_chunk,
    structured_chunk,
)
from easyrag.support.optional_deps import Document


def describe_chunks(name: str, chunks: list[Document]) -> None:
    print(f"=== {name} ({len(chunks)} chunks) ===")
    for chunk in chunks[:6]:
        preview = chunk.page_content[:140].replace("\n", " ")
        print(preview)
        pprint(chunk.metadata)
        print()
    if len(chunks) > 6:
        print(f"... omitted {len(chunks) - 6} additional chunks for readability.")

## What this cell is proving

We will push the chunk sizes down so the boundaries stay visible on a tiny corpus. This is not a production configuration. It is a teaching configuration.

In [ ]:
config = ChunkingConfig(
    sliding_window_size=90,
    sliding_window_overlaps=(20, 30, 40),
    structured_max_section_chars=80,
    structured_secondary_overlap=20,
    semantic_target_chars=80,
    semantic_sentence_overlap=1,
    semantic_similarity_threshold=0.85,
)

markdown_document = Document(
    page_content=(
        "# Overview\nEasyRAG organizes repository knowledge into stable retrieval units.\n"
        "## Retrieval\nQuery rewrite and hybrid retrieval keep the candidate set inspectable.\n"
        "## Storage\nPacked evidence remains traceable after generation.\n"
    ),
    metadata={
        "doc_id": "doc::markdown",
        "path": "docs/architecture.md",
        "title": "Architecture",
    },
)
plain_text_document = Document(
    page_content=(
        "EasyRAG stores summaries beside chunks. Sliding windows are simple to reason about. "
        "They ignore heading structure, but they are stable when the source format is messy. "
        "Rerank and metadata filters still operate on the resulting chunks."
    ),
    metadata={
        "doc_id": "doc::plain",
        "path": "notes/pipeline.txt",
        "title": "Pipeline Notes",
    },
)
pdf_like_document = Document(
    page_content=(
        "Sentence one explains retrieval behavior. Sentence two describes rewrite and rerank. "
        "Sentence three shifts to storage and answer packing. Sentence four summarizes citation-aware output."
    ),
    metadata={
        "doc_id": "doc::pdf",
        "path": "docs/architecture.pdf",
        "title": "Architecture PDF",
    },
)

print("Default strategy choices")
pprint(
    {
        markdown_document.metadata["path"]: select_chunk_strategy(markdown_document),
        plain_text_document.metadata["path"]: select_chunk_strategy(
            plain_text_document
        ),
        pdf_like_document.metadata["path"]: select_chunk_strategy(pdf_like_document),
    }
)

## Why this output looks like this

The next cell runs all three chunkers directly. Compare both the text boundaries and the metadata. The metadata is part of the teaching story because later retrieval and citations depend on it.

In [ ]:
describe_chunks(
    "structured on markdown", structured_chunk(markdown_document, config=config)
)
describe_chunks(
    "sliding window on markdown", sliding_window_chunk(markdown_document, config=config)
)
describe_chunks(
    "semantic on PDF-like text",
    semantic_chunk(pdf_like_document, config=config, embedding_func=_stub_embedding),
)
describe_chunks(
    "sliding window on plain text",
    sliding_window_chunk(plain_text_document, config=config),
)

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

Semantic chunking can also use the embedding function exposed by a provider-backed `EasyRAG` instance. The cell keeps the same document and config, then swaps in the real embedding function when the environment is configured.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_rag = EasyRAG(workspace="provider-chunking-demo")
    try:
        provider_chunks = semantic_chunk(
            pdf_like_document, config=config, embedding_func=provider_rag.embedding_func
        )
        describe_chunks("provider-backed semantic chunking", provider_chunks)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")

## Common mistakes / debugging cues

- Do not tune retrieval before you understand chunk boundaries, embedding inputs, and backend choice.
- Watch metadata such as `chunk_strategy`, `vector_backend`, and workspace artifacts, not only final query hits.
- Indexing bugs often look like retrieval bugs until you inspect the payloads that were actually written.

## Takeaway

The main difference across strategies is which boundary signal they trust. Sliding windows trust length. Structured chunking trusts headings first and falls back to windows when a section gets too large. Semantic chunking trusts embedding similarity between sentences. EasyRAG chooses a default strategy from metadata hints, but the chunk metadata always records what actually happened.

## Where to go next

- Continue with [03_04_normalization_before_embedding.ipynb](03_04_normalization_before_embedding.ipynb) to see why text cleanup still matters before semantic or dense stages.
- Read [03-indexing-overview.md](../../docs/03-indexing-overview.md) for the full indexing-stage narrative.
- Compare this notebook with [03_07_build_index_pipeline.ipynb](03_07_build_index_pipeline.ipynb) once you want the full insert-pipeline view.